# Test set preparation

This notebook gets the Test set (`BATADAL_test_dataset.csv`) ready for the final scoring. It comes after the data pipeline (`Pipeline.ipynb`, `docs/pipeline.md`).

- The Test file has no label column. So we build the labels from the 7 Test attacks in Table 3 of Taormina et al. (2018), the BATADAL organizers' paper. These are attacks 8 to 14, with 407 attack hours in total.
- We prepare the Test set the same way as the other files: round to 2 decimals, drop the same 7 columns, and scale with `scaler.joblib` (transform only, no new fit).

**This file is locked.** No model should be scored on `test_eval.csv` until we freeze all choices and thresholds. We don't use it to pick a model, a threshold or a setting.

Steps:
1. Load, round, drop, scale, label and save the Test set

The attack times for attacks 8-14 were checked against Taormina et al. (2018), Table 3.

## 1. Load, round, drop, scale, label and save

Why:
- The models are trained on rounded, scaled data without the 7 never-changing columns. The Test set must go through the same steps, or the models would see different numbers for the same reading.
- We only transform with the saved scaler. Fitting it again on the Test set would let Test information leak into the scaling.
- The Test file has no labels, so we need the attack list from the paper to score anything on it.

What the code does:
- Loads `../data/BATADAL_test_dataset.csv`, strips the spaces from the column names, and turns `DATETIME` into real dates (format day/month/year hour).
- Rounds every column except `DATETIME` to 2 decimals.
- Drops the same 7 columns as `Pipeline.ipynb` (`drop_cols`): `S_PU1`, `F_PU3`, `S_PU3`, `F_PU5`, `S_PU5`, `F_PU9` and `S_PU9`.
- Loads `scaler.joblib` and scales the 28 continuous columns with it (`scaler.transform`, no fit). The on/off columns are not touched.
- Builds a table of the 7 Test attacks (`attacks_test`): `attack_id`, `real_start` and `real_end` (both hours included), and `real_hours`.
- Adds two columns:
  - `label`: 1 if the hour is inside any of the 7 attacks, 0 otherwise.
  - `attack_id`: which attack the hour belongs to (8 to 14), or 0 if none.
- Prints the shape, the attack table and the counts, then saves `data/attack_windows_test.csv` and `data/processed/test_eval.csv`.

In [1]:
import pandas as pd
import joblib

test = pd.read_csv("../data/BATADAL_test_dataset.csv")
test.columns = test.columns.str.strip()
test["DATETIME"] = pd.to_datetime(test["DATETIME"], format="%d/%m/%y %H")

num_cols = [c for c in test.columns if c != "DATETIME"]
test[num_cols] = test[num_cols].round(2)

drop_cols = ["S_PU1", "F_PU3", "S_PU3", "F_PU5", "S_PU5", "F_PU9", "S_PU9"]
test = test.drop(columns=drop_cols)

scaler = joblib.load("../data/processed/scaler.joblib")
cont_cols = list(scaler.feature_names_in_)
test[cont_cols] = scaler.transform(test[cont_cols])

attacks_test = pd.DataFrame({
    "attack_id": [8, 9, 10, 11, 12, 13, 14],
    "real_start": pd.to_datetime(["2017-01-16 09:00", "2017-01-30 08:00", "2017-02-09 03:00",
                                  "2017-02-12 01:00", "2017-02-24 05:00", "2017-03-10 14:00",
                                  "2017-03-25 20:00"]),
    "real_end": pd.to_datetime(["2017-01-19 06:00", "2017-02-02 00:00", "2017-02-10 09:00",
                                "2017-02-13 07:00", "2017-02-28 08:00", "2017-03-13 21:00",
                                "2017-03-27 01:00"]),
})
attacks_test["real_hours"] = (attacks_test["real_end"] - attacks_test["real_start"]).dt.total_seconds() / 3600 + 1

test["label"] = 0
test["attack_id"] = 0
for _, a in attacks_test.iterrows():
    inside = (test["DATETIME"] >= a["real_start"]) & (test["DATETIME"] <= a["real_end"])
    test.loc[inside, "label"] = 1
    test.loc[inside, "attack_id"] = a["attack_id"]

print(test.shape)
print(attacks_test)
print(test["label"].value_counts())
print(test["attack_id"].value_counts().sort_index())

attacks_test.to_csv("../data/attack_windows_test.csv", index=False)
test.to_csv("../data/processed/test_eval.csv", index=False)

(2089, 39)
   attack_id          real_start            real_end  real_hours
0          8 2017-01-16 09:00:00 2017-01-19 06:00:00        70.0
1          9 2017-01-30 08:00:00 2017-02-02 00:00:00        65.0
2         10 2017-02-09 03:00:00 2017-02-10 09:00:00        31.0
3         11 2017-02-12 01:00:00 2017-02-13 07:00:00        31.0
4         12 2017-02-24 05:00:00 2017-02-28 08:00:00       100.0
5         13 2017-03-10 14:00:00 2017-03-13 21:00:00        80.0
6         14 2017-03-25 20:00:00 2017-03-27 01:00:00        30.0
label
0    1682
1     407
Name: count, dtype: int64
attack_id
0     1682
8       70
9       65
10      31
11      31
12     100
13      80
14      30
Name: count, dtype: int64


**What this shows:**
- The Test set has 2,089 hours and 39 columns: `DATETIME`, 36 features, `label` and `attack_id`. No rows were removed.
- `label`: 1,682 hours are normal (0) and 407 are attack (1).
- Hours per attack: attack 8 has 70, attack 9 has 65, attacks 10 and 11 have 31 each, attack 12 has 100, attack 13 has 80 and attack 14 has 30. These match `real_hours` in the attack table, so every attack hour is in the file.

Caution:
- Attacks 10 and 11 turn on pump 3. We dropped `F_PU3` and `S_PU3` because they never change in Dataset 1 or Dataset 2 (`docs/audit.md`), so the models can't see pump 3 running. They can only see its side effects on the other sensors. We did not change the column decision because of this. Changing it after looking at the Test attacks would mean using the Test set to make a choice.
- Hours outside the 7 attacks are not confirmed normal. We are trusting that the paper's list of attacks is complete.
- Detection delay should be counted from each attack's first hour in `attack_id`, which is its real start.

## End of notebook

The code cell below is empty.

Saved:
- `data/processed/test_eval.csv`: the prepared Test set (2,089 rows, 39 columns).
- `data/attack_windows_test.csv`: the 7 Test attacks with their start, end and hours.

Reminder: the Test set is locked. We only score models on it after all choices and thresholds are frozen.